# DistilBERT + SST-2 — Full DP Placement Analysis

**Model:** `distilbert-base-uncased` | **Task:** SST-2 (binary sentiment) | **δ = 1e-5**

**Placements:** no_dp · adapter_only · head_adapter · partial_backbone · last_layer · full_dp  
**PEFT methods:** Adapter, LoRA  
**ε values:** 0.5 · 1 · 2 · 4 · 8 · ∞ — new ε values appear automatically when the sweep runs

Figures saved to `results/figures_distilbert/`.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.4)
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.family': 'sans-serif', 'axes.spines.top': False, 'axes.spines.right': False,
})

PLACEMENT_ORDER = ['no_dp', 'adapter_only', 'head_adapter', 'partial_backbone', 'last_layer', 'full_dp']
DP_PLACEMENTS   = [p for p in PLACEMENT_ORDER if p != 'no_dp']

LABELS = {
    'no_dp':            'No DP (Baseline)',
    'adapter_only':     'Adapter-Only DP',
    'head_adapter':     'Head+Adapter DP',
    'partial_backbone': 'Partial Backbone DP',
    'last_layer':       'Last-Layer DP',
    'full_dp':          'Full-Model DP',
}
COLORS = {
    'no_dp':            '#555555',
    'adapter_only':     '#2ca02c',
    'head_adapter':     '#1f77b4',
    'partial_backbone': '#ff7f0e',
    'last_layer':       '#9467bd',
    'full_dp':          '#d62728',
}

RESULTS_ROOT = Path('../distilbert_results')
FIGS = Path('../results/figures_distilbert')
FIGS.mkdir(parents=True, exist_ok=True)
print('Setup done. Figures ->', FIGS.resolve())

In [ ]:
## Load all training JSONs + MIA logs (auto-discovers all ε values)

def load_training_results(root):
    rows = []
    for peft in ('adapter', 'lora'):
        pdir = root / peft
        if not pdir.exists():
            continue
        for jf in sorted(pdir.glob('distilbert_sst2_*.json')):
            stem   = jf.stem
            prefix = f'distilbert_sst2_{peft}_'
            if not stem.startswith(prefix):
                continue
            rest = stem[len(prefix):]
            if '_eps' not in rest:
                continue
            idx       = rest.rfind('_eps')
            placement = rest[:idx]
            eps_str   = rest[idx + 4:]
            try:
                epsilon = float(eps_str)
            except ValueError:
                continue
            try:
                raw = jf.read_text().replace('Infinity', '1e38')
                d   = json.loads(raw)
            except Exception as e:
                print(f'  SKIP {jf.name}: {e}')
                continue
            rows.append({
                'peft': peft, 'placement': placement, 'epsilon': epsilon,
                'accuracy':   d.get('final_accuracy'),
                'f1':         d.get('final_f1'),
                'epoch_time': d.get('avg_epoch_time'),
                'throughput': d.get('avg_throughput'),
                'grad_var':   d.get('grad_norm_variance'),
                'loss_osc':   d.get('loss_oscillation'),
                'acc_curve':  d.get('accuracy_curve', []),
                'loss_curve': d.get('convergence_curve', []),
            })
    return pd.DataFrame(rows)


def load_mia_results(root):
    rows = []
    for peft in ('adapter', 'lora'):
        pdir = root / peft
        if not pdir.exists():
            continue
        for logf in sorted(pdir.glob('mia_*.log')):
            inner = logf.stem[4:]          # strip 'mia_'
            if '_eps' not in inner:
                continue
            idx       = inner.rfind('_eps')
            placement = inner[:idx]
            rest      = inner[idx + 4:]
            eps_str   = rest.split('_')[0]
            try:
                epsilon = float(eps_str)
            except ValueError:
                continue
            auc, adv = None, None
            try:
                for line in logf.read_text(errors='ignore').splitlines():
                    if 'Threshold Attack AUC:' in line:
                        auc = float(line.split(':')[-1].strip())
                    elif 'Threshold Attack Advantage:' in line:
                        adv = float(line.split(':')[-1].strip())
            except Exception:
                continue
            if auc is None:
                continue
            rows.append({'peft': peft, 'placement': placement,
                         'epsilon': epsilon, 'auc': auc, 'advantage': adv})
    return pd.DataFrame(rows)


train_df = load_training_results(RESULTS_ROOT)
mia_df   = load_mia_results(RESULTS_ROOT)

print(f'Training results : {len(train_df)} rows')
print(f'  PEFT methods   : {sorted(train_df.peft.unique())}')
print(f'  ε values       : {sorted(train_df.epsilon.unique())}')
print(f'MIA results      : {len(mia_df)} rows')
print()
pivot = train_df.pivot_table(
    index=['peft','placement'], columns='epsilon', values='accuracy', aggfunc='first'
).round(4)
print(pivot.to_string())

In [ ]:
## Figure 1 — Privacy-Utility Curve (ε sweep, adapter PEFT)
# Points at ε={0.5,2,4} appear automatically once the sweep results land in distilbert_results/adapter/

fig, ax = plt.subplots(figsize=(11, 6))

df_a = train_df[train_df.peft == 'adapter'].copy()
nodp = df_a[df_a.placement == 'no_dp']
nodp_acc = float(nodp.iloc[0]['accuracy']) if not nodp.empty else None

for p in DP_PLACEMENTS:
    pdata = df_a[df_a.placement == p].sort_values('epsilon')
    if pdata.empty:
        continue
    eps_vals = pdata['epsilon'].values
    acc_vals = pdata['accuracy'].values * 100
    ax.plot(eps_vals, acc_vals, color=COLORS[p], marker='o',
            linewidth=2.2, markersize=8, label=LABELS[p])
    ax.annotate(f'{acc_vals[-1]:.1f}%', xy=(eps_vals[-1], acc_vals[-1]),
                xytext=(5, 3), textcoords='offset points', fontsize=8.5, color=COLORS[p])

if nodp_acc is not None:
    ax.axhline(nodp_acc * 100, color=COLORS['no_dp'], linestyle='--',
               linewidth=2, alpha=0.8, label=f'No DP ceiling ({nodp_acc:.1%})')

ax.set_xscale('log')
ax.set_xticks([0.5, 1, 2, 4, 8])
ax.set_xticklabels(['0.5', '1', '2', '4', '8'], fontsize=11)
ax.set_xlabel('Privacy Budget  ε   (larger = weaker privacy)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title(
    'Privacy-Utility Curve by DP Placement Strategy\n'
    'DistilBERT-base · SST-2 · Adapter PEFT · δ=1e-5',
    fontsize=13, fontweight='bold'
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}%'))
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, which='both', alpha=0.3)

ylims = ax.get_ylim()
ax.axvspan(0.4, 1.5, alpha=0.07, color='blue')
ax.text(0.42, ylims[0] + 0.5, 'Strong\nprivacy\nzone', fontsize=8, color='blue', alpha=0.7)

plt.tight_layout()
plt.savefig(FIGS / 'fig1_privacy_utility_curve.pdf')
plt.savefig(FIGS / 'fig1_privacy_utility_curve.png')
plt.show()
print('Saved fig1_privacy_utility_curve')

In [ ]:
## Figure 2 — Accuracy by Placement: both PEFT methods, ε={1,8}

def get_acc(p, peft, eps):
    row = train_df[(train_df.placement==p) & (train_df.peft==peft) & (train_df.epsilon==eps)]
    if row.empty and p == 'no_dp':
        row = train_df[(train_df.placement==p) & (train_df.peft==peft)]
    return float(row.iloc[0]['accuracy']) * 100 if not row.empty else np.nan

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
for ax, eps_val in zip(axes, [8.0, 1.0]):
    x  = np.arange(len(PLACEMENT_ORDER))
    bw = 0.36
    acc_adapter = [get_acc(p, 'adapter', eps_val) for p in PLACEMENT_ORDER]
    acc_lora    = [get_acc(p, 'lora',    eps_val) for p in PLACEMENT_ORDER]

    bars_a = ax.bar(x - bw/2, acc_adapter, bw,
                    color=[COLORS[p] for p in PLACEMENT_ORDER], alpha=0.92, edgecolor='white')
    bars_l = ax.bar(x + bw/2, acc_lora, bw,
                    color=[COLORS[p] for p in PLACEMENT_ORDER], alpha=0.52, edgecolor='white', hatch='//')

    nodp_a = acc_adapter[0]
    if not np.isnan(nodp_a):
        ax.axhline(nodp_a, color=COLORS['no_dp'], linestyle='--', linewidth=1.5, alpha=0.5)

    for bars in (bars_a, bars_l):
        for bar in bars:
            v = bar.get_height()
            if not np.isnan(v):
                ax.text(bar.get_x()+bar.get_width()/2, v+0.1,
                        f'{v:.1f}', ha='center', va='bottom', fontsize=7, rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels([LABELS[p] for p in PLACEMENT_ORDER], rotation=22, ha='right', fontsize=10)
    ax.set_ylabel('Test Accuracy (%)', fontsize=11)
    ax.set_title(f'ε = {eps_val:.0f}  ({"loose" if eps_val==8 else "tight"} privacy)',
                 fontsize=12, fontweight='bold')
    ax.set_ylim(74, 95)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))

    solid   = mpatches.Patch(facecolor='gray', alpha=0.92, label='Adapter')
    hatched = mpatches.Patch(facecolor='gray', alpha=0.52, hatch='//', label='LoRA')
    ax.legend(handles=[solid, hatched], fontsize=10, loc='lower right')

fig.suptitle('Classification Accuracy by DP Placement  (DistilBERT · SST-2 · 20 epochs)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'fig2_accuracy_by_placement.pdf')
plt.savefig(FIGS / 'fig2_accuracy_by_placement.png')
plt.show()
print('Saved fig2_accuracy_by_placement')

In [ ]:
## Figure 3 — LoRA vs Adapter Scatter

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, eps_val in zip(axes, [8.0, 1.0]):
    for p in PLACEMENT_ORDER:
        ra = train_df[(train_df.placement==p) & (train_df.peft=='adapter') & (train_df.epsilon==eps_val)]
        rl = train_df[(train_df.placement==p) & (train_df.peft=='lora')    & (train_df.epsilon==eps_val)]
        if p == 'no_dp':
            ra = train_df[(train_df.placement==p) & (train_df.peft=='adapter')]
            rl = train_df[(train_df.placement==p) & (train_df.peft=='lora')]
        if ra.empty or rl.empty:
            continue
        xa = float(ra.iloc[0]['accuracy']) * 100
        yl = float(rl.iloc[0]['accuracy']) * 100
        ax.scatter(xa, yl, color=COLORS[p], s=130, zorder=3, edgecolors='white', linewidths=0.7)
        ax.annotate(LABELS[p], xy=(xa, yl), xytext=(4, 2),
                    textcoords='offset points', fontsize=8, color=COLORS[p])

    lims = [74, 95]
    ax.plot(lims, lims, 'k--', linewidth=1.2, alpha=0.4, label='LoRA = Adapter')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Adapter Accuracy (%)', fontsize=11)
    ax.set_ylabel('LoRA Accuracy (%)', fontsize=11)
    ax.set_title(f'LoRA vs Adapter  (ε = {eps_val:.0f})', fontsize=12, fontweight='bold')
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
    ax.legend(fontsize=9)
    ax.text(lims[0]+0.2, lims[1]-1.5,
            'Above diagonal: LoRA wins\nBelow diagonal: Adapter wins',
            fontsize=8, color='gray')

fig.suptitle('PEFT Method Comparison: LoRA vs Adapter  (DistilBERT · SST-2)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'fig3_lora_vs_adapter.pdf')
plt.savefig(FIGS / 'fig3_lora_vs_adapter.png')
plt.show()
print('Saved fig3_lora_vs_adapter')

In [ ]:
## Figure 4 — MIA Results (adapter + LoRA, both ε)

def get_mia(p, peft, eps, metric):
    row = mia_df[(mia_df.placement==p) & (mia_df.peft==peft) & (mia_df.epsilon==eps)]
    if row.empty and p == 'no_dp':
        row = mia_df[(mia_df.placement==p) & (mia_df.peft==peft)]
    return float(row.iloc[0][metric]) if not row.empty else np.nan

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for ri, peft in enumerate(('adapter', 'lora')):
    for ci, (metric, ylabel, ref, ref_lbl) in enumerate([
        ('auc',       'MIA Attack AUC',   0.50, 'Random guess (0.50)'),
        ('advantage', 'MIA Advantage',    0.00, 'Ideal DP (0.00)'),
    ]):
        ax = axes[ri][ci]
        x  = np.arange(len(PLACEMENT_ORDER))
        bw = 0.36

        v8 = [get_mia(p, peft, 8.0, metric) for p in PLACEMENT_ORDER]
        v1 = [get_mia(p, peft, 1.0, metric) for p in PLACEMENT_ORDER]

        b8 = ax.bar(x - bw/2, v8, bw, color=[COLORS[p] for p in PLACEMENT_ORDER],
                    alpha=0.92, edgecolor='white')
        b1 = ax.bar(x + bw/2, v1, bw, color=[COLORS[p] for p in PLACEMENT_ORDER],
                    alpha=0.52, edgecolor='white', hatch='//')

        ax.axhline(ref, color='black', linestyle='--', linewidth=1.4)
        ax.set_xticks(x)
        ax.set_xticklabels([LABELS[p] for p in PLACEMENT_ORDER],
                           rotation=25, ha='right', fontsize=9)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(f'{peft.upper()}  —  {ylabel}', fontsize=11, fontweight='bold')

        for bar, val in zip(list(b8)+list(b1), v8+v1):
            if not np.isnan(val):
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0005,
                        f'{val:.4f}', ha='center', va='bottom', fontsize=6.5, rotation=90)

        solid   = mpatches.Patch(facecolor='gray', alpha=0.92, label='ε = 8.0')
        hatched = mpatches.Patch(facecolor='gray', alpha=0.52, hatch='//', label='ε = 1.0')
        ref_patch = mpatches.Patch(facecolor='white', edgecolor='black',
                                   linestyle='--', label=ref_lbl)
        ax.legend(handles=[solid, hatched, ref_patch], fontsize=8, loc='upper right')

fig.suptitle('Membership Inference Attack Results  (DistilBERT · SST-2)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGS / 'fig4_mia_results.pdf')
plt.savefig(FIGS / 'fig4_mia_results.png')
plt.show()
print('Saved fig4_mia_results')

In [ ]:
## Figure 5 — Training Convergence (adapter, ε=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

df_a1  = train_df[(train_df.peft == 'adapter') & (train_df.epsilon == 1.0)]
nodp_r = train_df[(train_df.peft == 'adapter') & (train_df.placement == 'no_dp')]

for p in PLACEMENT_ORDER:
    row = nodp_r if p == 'no_dp' else df_a1[df_a1.placement == p]
    if row.empty:
        continue
    c   = COLORS[p]
    lbl = LABELS[p]
    ls  = '--' if p == 'no_dp' else '-'
    lw  = 2.2  if p in ('no_dp', 'adapter_only') else 1.7

    lc = row.iloc[0]['loss_curve']
    ac = row.iloc[0]['acc_curve']
    if lc:
        ax1.plot(range(1, len(lc)+1), lc, color=c, lw=lw, ls=ls, label=lbl)
    if ac:
        ax2.plot(range(1, len(ac)+1), [v*100 for v in ac], color=c, lw=lw, ls=ls, label=lbl)

ax1.set_xlabel('Epoch', fontsize=11); ax1.set_ylabel('Training Loss', fontsize=11)
ax1.set_title('Loss Convergence  (ε = 1.0)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9, loc='upper right')

ax2.set_xlabel('Epoch', fontsize=11); ax2.set_ylabel('Test Accuracy (%)', fontsize=11)
ax2.set_title('Accuracy Curve  (ε = 1.0)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9, loc='lower right')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))

fig.suptitle('Training Dynamics by DP Placement  (DistilBERT · SST-2 · Adapter)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'fig5_convergence.pdf')
plt.savefig(FIGS / 'fig5_convergence.png')
plt.show()
print('Saved fig5_convergence')

In [ ]:
## Figure 6 — Stability & Efficiency

fig, axes = plt.subplots(1, 3, figsize=(17, 6))

combos   = [('adapter', 1.0), ('adapter', 8.0), ('lora', 1.0), ('lora', 8.0)]
clabels  = ['Adapter ε=1', 'Adapter ε=8', 'LoRA ε=1', 'LoRA ε=8']
ccolors  = ['#1f77b4', '#aec7e8', '#ff7f0e', '#ffbb78']
chatches = ['', '', '//', '//']
bw = 0.2
x  = np.arange(len(PLACEMENT_ORDER))

for ax, (col, ylabel, title) in zip(axes, [
    ('grad_var',   'Gradient Norm Variance',    'Stability: Gradient Variance'),
    ('epoch_time', 'Avg Epoch Time (s)',         'Efficiency: Epoch Time'),
    ('throughput', 'Throughput (samples / sec)', 'Efficiency: Throughput'),
]):
    for gi, (peft, eps) in enumerate(combos):
        vals = []
        for p in PLACEMENT_ORDER:
            row = train_df[(train_df.placement==p) & (train_df.peft==peft) & (train_df.epsilon==eps)]
            if row.empty and p == 'no_dp':
                row = train_df[(train_df.placement==p) & (train_df.peft==peft)]
            vals.append(float(row.iloc[0][col]) if not row.empty else np.nan)
        offset = (gi - 1.5) * bw
        ax.bar(x + offset, vals, bw, color=ccolors[gi], alpha=0.88,
               edgecolor='white', hatch=chatches[gi], label=clabels[gi])

    ax.set_xticks(x)
    ax.set_xticklabels([LABELS[p] for p in PLACEMENT_ORDER], rotation=25, ha='right', fontsize=8.5)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=7.5, ncol=2)

fig.suptitle('Training Stability and Efficiency by DP Placement  (DistilBERT · SST-2)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'fig6_stability_efficiency.pdf')
plt.savefig(FIGS / 'fig6_stability_efficiency.png')
plt.show()
print('Saved fig6_stability_efficiency')

In [ ]:
## Figure 7 — Utility vs Privacy Frontier (Accuracy vs MIA Advantage)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, peft in zip(axes, ('adapter', 'lora')):
    merged = pd.merge(
        train_df[train_df.peft == peft][['placement','epsilon','accuracy']],
        mia_df[mia_df.peft == peft][['placement','epsilon','advantage']],
        on=['placement','epsilon'], how='inner'
    )
    # Also add no_dp (epsilon-independent)
    nodp_tr  = train_df[(train_df.placement=='no_dp') & (train_df.peft==peft)]
    nodp_mia = mia_df[(mia_df.placement=='no_dp') & (mia_df.peft==peft)]
    if not nodp_tr.empty and not nodp_mia.empty:
        nodp_row = pd.DataFrame([{
            'placement': 'no_dp',
            'epsilon': nodp_mia.iloc[0]['epsilon'],
            'accuracy': float(nodp_tr.iloc[0]['accuracy']),
            'advantage': float(nodp_mia.iloc[0]['advantage']),
        }])
        merged = pd.concat([merged, nodp_row], ignore_index=True)

    for _, row in merged.iterrows():
        p   = row['placement']
        eps = row['epsilon']
        ax.scatter(row['advantage'], row['accuracy']*100,
                   color=COLORS[p],
                   s=160 if eps == 8.0 else 90,
                   marker='o' if eps == 8.0 else 's',
                   zorder=3, edgecolors='white', linewidths=0.7)
        ax.annotate(f'{LABELS[p]}\n(ε={"8" if eps==8.0 else "1" if eps==1.0 else str(eps)})',
                    xy=(row['advantage'], row['accuracy']*100),
                    xytext=(4, 2), textcoords='offset points',
                    fontsize=7, color=COLORS[p])

    ax.set_xlabel('MIA Advantage  (higher = less private)', fontsize=11)
    ax.set_ylabel('Test Accuracy (%)', fontsize=11)
    ax.set_title(f'{peft.upper()} PEFT — Privacy-Utility Frontier', fontsize=12, fontweight='bold')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
    ax.grid(True, alpha=0.3)

    circle_8 = plt.Line2D([0],[0], marker='o', color='gray', ls='None', ms=10, label='ε = 8.0')
    circle_1 = plt.Line2D([0],[0], marker='s', color='gray', ls='None', ms=8,  label='ε = 1.0')
    ax.legend(handles=[circle_8, circle_1], fontsize=9)
    ax.text(0.01, ax.get_ylim()[0]+0.5,
            '← better privacy\n↑ better utility\n(ideal: top-left)',
            fontsize=8, color='green', alpha=0.8)

fig.suptitle('Privacy-Utility Frontier  (DistilBERT · SST-2)\n'
             '●=ε 8.0   ■=ε 1.0', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'fig7_utility_privacy_frontier.pdf')
plt.savefig(FIGS / 'fig7_utility_privacy_frontier.png')
plt.show()
print('Saved fig7_utility_privacy_frontier')

In [ ]:
## Summary Table + LaTeX

nodp_adapter = train_df[(train_df.placement=='no_dp') & (train_df.peft=='adapter')]
nodp_lora    = train_df[(train_df.placement=='no_dp') & (train_df.peft=='lora')]
baseline_a   = float(nodp_adapter.iloc[0]['accuracy']) if not nodp_adapter.empty else None
baseline_l   = float(nodp_lora.iloc[0]['accuracy'])    if not nodp_lora.empty    else None

rows_tbl = []
for p in PLACEMENT_ORDER:
    for peft, baseline in (('adapter', baseline_a), ('lora', baseline_l)):
        for eps in ([None] if p == 'no_dp' else [8.0, 1.0]):
            if p == 'no_dp':
                tr = train_df[(train_df.placement==p) & (train_df.peft==peft)]
                mi = mia_df[(mia_df.placement==p) & (mia_df.peft==peft)]
            else:
                tr = train_df[(train_df.placement==p) & (train_df.peft==peft) & (train_df.epsilon==eps)]
                mi = mia_df[(mia_df.placement==p) & (mia_df.peft==peft) & (mia_df.epsilon==eps)]
            if tr.empty:
                continue
            acc  = float(tr.iloc[0]['accuracy'])
            drop = (baseline - acc) if baseline is not None else None
            rows_tbl.append({
                'PEFT':    peft,
                'Placement': LABELS[p],
                'eps':     '—' if p == 'no_dp' else f'{eps:.0f}',
                'Accuracy': f'{acc:.4f}',
                'D_NoDp':  f'-{drop:.3f}' if drop is not None else '—',
                'Epoch_s': f'{tr.iloc[0]["epoch_time"]:.0f}',
                'GradVar': f'{tr.iloc[0]["grad_var"]:.4f}',
                'MIA_AUC': f'{float(mi.iloc[0]["auc"]):.4f}'       if not mi.empty else '—',
                'MIA_Adv': f'{float(mi.iloc[0]["advantage"]):.4f}' if not mi.empty else '—',
            })
        if p == 'no_dp':
            break  # no_dp appears once per peft

tbl = pd.DataFrame(rows_tbl)
print(tbl.to_string(index=False))

# ── LaTeX output ────────────────────────────────────────────────────────────
col_map = {'PEFT':'PEFT','Placement':'Placement','eps':r'$\varepsilon$',
           'Accuracy':'Accuracy','D_NoDp':r'$\Delta$ No-DP',
           'Epoch_s':'Epoch (s)','GradVar':'Grad. Var.',
           'MIA_AUC':'MIA AUC','MIA_Adv':'MIA Adv.'}
cols = list(col_map.keys())
col_hdrs = list(col_map.values())

lines = [
    r'\begin{table}[h]', r'\centering',
    r'\caption{DistilBERT-base on SST-2. $\delta=10^{-5}$ for all DP runs.}',
    r'\label{tab:distilbert_results}',
    r'\small',
    r'\begin{tabular}{llc' + 'c'*(len(cols)-3) + '}',
    r'\hline',
    ' & '.join(col_hdrs) + r' \\',
    r'\hline',
]
for _, row in tbl.iterrows():
    lines.append(' & '.join(str(row[c]) for c in cols) + r' \\')
lines += [r'\hline', r'\end{tabular}', r'\end{table}']
latex = '\n'.join(lines)
print('\n\n=== LaTeX ===\n')
print(latex)

(Path('../results') / 'distilbert_summary_table.tex').write_text(latex)
print('\nSaved to results/distilbert_summary_table.tex')

In [ ]:
## Key Quantitative Findings

def g_acc(p, peft, eps=8.0):
    row = train_df[(train_df.placement==p) & (train_df.peft==peft) & (train_df.epsilon==eps)]
    if row.empty: row = train_df[(train_df.placement==p) & (train_df.peft==peft)]
    return float(row.iloc[0]['accuracy']) if not row.empty else None

def g_mia(p, peft, eps, metric):
    row = mia_df[(mia_df.placement==p) & (mia_df.peft==peft) & (mia_df.epsilon==eps)]
    if row.empty: row = mia_df[(mia_df.placement==p) & (mia_df.peft==peft)]
    return float(row.iloc[0][metric]) if not row.empty else None

# Adapter PEFT
nd_a  = g_acc('no_dp',        'adapter')
ao8_a = g_acc('adapter_only', 'adapter', 8.0)
ao1_a = g_acc('adapter_only', 'adapter', 1.0)
fd8_a = g_acc('full_dp',      'adapter', 8.0)
fd1_a = g_acc('full_dp',      'adapter', 1.0)
pb8_a = g_acc('partial_backbone', 'adapter', 8.0)
et_ao = float(train_df[(train_df.placement=='adapter_only')&(train_df.peft=='adapter')&(train_df.epsilon==8.0)].iloc[0]['epoch_time'])
et_fd = float(train_df[(train_df.placement=='full_dp')&(train_df.peft=='adapter')&(train_df.epsilon==8.0)].iloc[0]['epoch_time'])

# LoRA PEFT
nd_l  = g_acc('no_dp',        'lora')
ao8_l = g_acc('adapter_only', 'lora', 8.0)
ao1_l = g_acc('adapter_only', 'lora', 1.0)
fd8_l = g_acc('full_dp',      'lora', 8.0)

print('=' * 65)
print('KEY QUANTITATIVE FINDINGS — DistilBERT + SST-2')
print('=' * 65)
print()
print('--- UTILITY ---')
print(f' No-DP baseline (adapter)          : {nd_a:.2%}')
print(f' No-DP baseline (LoRA)             : {nd_l:.2%}')
print(f' Adapter-Only ε=8 (adapter)        : {ao8_a:.2%}  (drop: {nd_a-ao8_a:.2%})')
print(f' Adapter-Only ε=8 (LoRA)           : {ao8_l:.2%}  (drop: {nd_l-ao8_l:.2%})')
print(f' Adapter-Only ε=1 (adapter)        : {ao1_a:.2%}  (drop: {nd_a-ao1_a:.2%})')
print(f' Adapter-Only ε=1 (LoRA)           : {ao1_l:.2%}  (drop: {nd_l-ao1_l:.2%})')
print(f' ε sensitivity adapter-only        : {ao8_a-ao1_a:.2%} (adapter)  {ao8_l-ao1_l:.2%} (LoRA)')
print(f' Full-DP ε=8 (adapter)             : {fd8_a:.2%}  (drop: {nd_a-fd8_a:.2%})')
print(f' Full-DP ε=8 (LoRA)               : {fd8_l:.2%}  (drop: {nd_l-fd8_l:.2%})')
print(f' Adapter-Only vs Full-DP gap ε=8   : {ao8_a-fd8_a:.2%}  <- core thesis')
print()
print('--- EFFICIENCY ---')
print(f' Epoch time: adapter_only={et_ao:.0f}s  full_dp={et_fd:.0f}s  ({et_fd/et_ao:.1f}x slower)')
print()
print('--- EMPIRICAL PRIVACY (MIA) ---')
for peft in ('adapter', 'lora'):
    print(f'  {peft.upper()}:')
    for p in PLACEMENT_ORDER:
        auc = g_mia(p, peft, 8.0, 'auc')
        adv = g_mia(p, peft, 8.0, 'advantage')
        if auc is not None:
            note = ' ← best' if (auc is not None and auc < 0.50) else ''
            note = note or (' ← WORSE than no_dp!' if p == 'full_dp' and
                            g_mia('no_dp', peft, 8.0, 'auc') is not None and
                            auc > g_mia('no_dp', peft, 8.0, 'auc') else '')
            print(f'    {LABELS[p]:25s} AUC={auc:.4f}  Adv={adv:.4f}{note}')

## Key Findings

### 1. Core Thesis Confirmed
PEFT-based DP placements consistently outperform Full-Model DP by **7–11 pp** in accuracy at the same ε — across both Adapter and LoRA PEFT methods.

### 2. Best Accuracy-Privacy Tradeoff
**Adapter-Only DP** is the Pareto-dominant strategy:
- Adapter: only **−2.75%** accuracy drop at ε=8 vs no-DP baseline
- LoRA: only **−3.67%** at ε=8 — comparable
- Both are far ahead of Full-Model DP (−10–13%)

### 3. full_dp MIA Paradox (Novel Finding)
Full-Model DP achieves **higher MIA AUC than no_dp** (0.601 vs 0.597 for adapter). Excessive noise causes training divergence which creates stronger fingerprints on training samples — more DP noise ≠ better empirical privacy when training is unstable.

### 4. last_layer Achieves Best Empirical Privacy
MIA AUC **below 0.50** (0.4951) — the attacker performs worse than random. But at a significant accuracy cost (−7–8%). Not the recommended strategy for most use cases.

### 5. LoRA ≈ Adapter Under DP
Both PEFT methods behave nearly identically. LoRA shows slightly better ε-robustness (only 0.46% drop from ε=8→1 vs 1.72% for adapter).

### 6. Figures produced
| File | Content |
|---|---|
| `fig1_privacy_utility_curve` | Accuracy vs ε for all placements (auto-updates with sweep) |
| `fig2_accuracy_by_placement` | Bar chart, both PEFT, ε={1,8} |
| `fig3_lora_vs_adapter` | Scatter comparison of LoRA vs Adapter |
| `fig4_mia_results` | MIA AUC + Advantage, 2×2 grid |
| `fig5_convergence` | Loss + accuracy curves over 20 epochs |
| `fig6_stability_efficiency` | Grad variance, epoch time, throughput |
| `fig7_utility_privacy_frontier` | Accuracy vs MIA Advantage scatter |